# 02_1 — Base Diagnóstico

**Objetivo:** Construir uma base analítica interpretável para dashboards, derivada de `base_full_raw.parquet`
(universo completo gerado em `01_ingest`, antes dos filtros do pipeline do modelo).
A base diagnóstico utiliza os valores originais das features (sem as versões faixadas do modelo)
e aplica faixamento manual e interpretável via dicionário de features.

**Diferença em relação ao notebook 02:**
- Notebook 02 aplica faixamento automático por quintis sobre a base TT (já filtrada por `fg_sem_output==0`,
  `dropna(target)` e `fg_ativo==1`) para o modelo preditivo.
- Este notebook usa `base_full_raw.parquet`, que preserva **todas** as linhas da mensalizada
  (incluindo desligados e meses sem target válido), e aplica faixamento com cortes manuais interpretáveis
  para análise em dashboards.

---

| Entrada | Descrição |
|---|---|
| `data/gold/base_full_raw.parquet` | Universo completo (todas as linhas da mensalizada, com rolling features) |

| Saída | Descrição |
|---|---|
| `reports/features_diagnostico.xlsx` | Catálogo de features para preenchimento do dicionário |
| `data/gold/base_diagnostico.parquet` | Base diagnóstico faixada (todos os períodos, todos os colaboradores) |


## 1 · Setup & Configuração

In [29]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pandas {pd.__version__}  ·  numpy {np.__version__}")
print(f"Execução: {datetime.now():%Y-%m-%d %H:%M}")


pandas 2.3.3  ·  numpy 2.3.5
Execução: 2026-05-07 15:32


In [30]:
PROJECT_ROOT   = Path.cwd().parent
DATA_GOLD      = PROJECT_ROOT / "data" / "gold"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS        = PROJECT_ROOT / "reports"

GRUPOS    = ["Vendas", "Transporte", "Fábrica"]
COL_GRUPO = "ds_grupo"
COL_ID    = "id_colaborador"
COL_DATA  = "dt_mes_ano"

JANELA_PREDICAO = 4
COL_TARGET = f"fg_demitido_voluntario_{JANELA_PREDICAO}m"

# Padrão para identificar colunas meta (nunca entram como features)
META_RE = re.compile(r"^fg_|^id_|^dt_|^ds_grupo$")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_GOLD    : {DATA_GOLD}")
print(f"REPORTS      : {REPORTS}")


PROJECT_ROOT : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo
DATA_GOLD    : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold
REPORTS      : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\reports


## 2 · Identificação de Features

In [ ]:
# ────────────────────────────────────────────────────────────────
# Carrega base_full_raw.parquet — universo completo gerado em 01_ingest
# ANTES dos filtros do pipeline do modelo (fg_sem_output, dropna(target),
# fg_ativo==1). Mantém todas as linhas da mensalizada com as features
# rolling já calculadas. Uso exclusivo: diagnóstico/dashboard.
# ────────────────────────────────────────────────────────────────
df_base = pd.read_parquet(DATA_GOLD / "base_full_raw.parquet")

print(f"base_full_raw.parquet: {df_base.shape[0]:,} linhas × {df_base.shape[1]} colunas")
print(f"  Período: {pd.to_datetime(df_base[COL_DATA]).min():%Y-%m} → {pd.to_datetime(df_base[COL_DATA]).max():%Y-%m}")
print(f"  IDs: {df_base[COL_ID].nunique():,}")

# Colunas meta
meta_cols = [c for c in df_base.columns if META_RE.match(c)]

# Colunas _bin criadas em 02 para o modelo — não existem em base_full_raw,
# mas mantemos o filtro para robustez (caso a fonte mude no futuro).
cols_bin_modelo = [c for c in df_base.columns if c.endswith("_bin")]

# Features originais: tudo que não é meta e não é _bin
feature_cols = [
    c for c in df_base.columns
    if c not in meta_cols and not c.endswith("_bin")
]

print(f"\nColunas meta         : {len(meta_cols)}")
print(f"Colunas _bin (modelo): {len(cols_bin_modelo)}")
print(f"Features originais   : {len(feature_cols)}")
print(f"\nPrimeiras 20 features:")
for c in feature_cols[:20]:
    print(f"  {c}  [{df_base[c].dtype}]")
if len(feature_cols) > 20:
    print(f"  ... e mais {len(feature_cols) - 20}")


base_features.parquet: 106,902 linhas × 834 colunas
  Período: 2024-01 → 2025-05
  IDs: 11,211

Colunas meta         : 17
Colunas _bin (modelo): 391
Features originais   : 426

Primeiras 20 features:
  ds_cargo  [object]
  ds_uorg  [object]
  ds_centro_custo  [object]
  ds_genero  [object]
  ds_etnia  [object]
  ds_pcd_detalhe  [object]
  ds_pcd  [object]
  ds_grade  [object]
  ds_tipo_turno  [object]
  vl_tempo_funcao  [float64]
  ds_escolaridade  [object]
  vl_salario  [float64]
  vl_beneficio_tempo_1  [float64]
  vl_beneficio_tempo_2  [float64]
  vl_beneficio_extra_1  [float64]
  vl_beneficio_extra_2  [float64]
  vl_beneficio_extra_3  [float64]
  vl_beneficio_extra_4  [float64]
  vl_remuneracao_variavel  [float64]
  vl_beneficio_extra_5  [float64]
  ... e mais 406


## 3 · Catálogo de Features

In [32]:
# ════════════════════════════════════════════════════════════════
# Gera ou atualiza reports/features_diagnostico.xlsx com estatísticas
# descritivas de cada feature e campos para preenchimento manual:
#   - label              : nome interpretável para o dashboard
#   - tipo_feature       : contínua | binária | categórica | percentual | desvio_padrao
#   - cortes_sugeridos   : preenchido automaticamente (Seção 4.3) como ponto de partida
#   - cortes_finais      : preencher após análise — usado na Seção 5
#   - incluir_diagnostico: 0 para excluir da base diagnóstico
# ════════════════════════════════════════════════════════════════
cat_path = REPORTS / "features_diagnostico.xlsx"

stats = []
for c in feature_cols:
    s = df_base[c]
    is_num = pd.api.types.is_numeric_dtype(s)
    row = {
        "feature":               c,
        "dtype":                 str(s.dtype),
        "n_distintos":           s.nunique(dropna=True),
        "pct_nulo":              round(s.isna().mean() * 100, 1),
        "tem_zero":              bool((s == 0).any()) if is_num else False,
        "min":                   round(float(s.min()), 4) if is_num else None,
        "max":                   round(float(s.max()), 4) if is_num else None,
        "media":                 round(float(s.mean()), 4) if is_num else None,
        "mediana":               round(float(s.median()), 4) if is_num else None,
        "label":                 "",
        "tipo_feature":          "",
        "cortes_sugeridos":      "",
        "cortes_finais":         "",
        "incluir_diagnostico":   1,
    }
    stats.append(row)

cat_df = pd.DataFrame(stats)

if not cat_path.exists():
    cat_df.to_excel(cat_path, index=False)
    print(f"✓ Catálogo criado: {cat_path.name}  ({len(feature_cols)} features)")
else:
    existing = pd.read_excel(cat_path)
    existing_feats = set(existing["feature"].tolist())
    new_rows = [r for r in stats if r["feature"] not in existing_feats]
    if new_rows:
        updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
        updated.to_excel(cat_path, index=False)
        print(f"✓ Catálogo atualizado: +{len(new_rows)} novas features  ({len(updated)} total)")
    else:
        print(f"✓ Catálogo já existe e está atualizado  ({len(existing)} features)")
    cat_df = pd.read_excel(cat_path)

print(f"\n⚠  PRÓXIMA AÇÃO: abra {cat_path.name} e preencha as colunas:")
print(f"   - label              : nome interpretável para o dashboard")
print(f"   - tipo_feature       : contínua | binária | categórica | percentual | desvio_padrao")
print(f"   - cortes_finais      : valores de corte separados por vírgula (ex: 0, 1000, 3000)")
print(f"   - incluir_diagnostico: 0 para excluir da base diagnóstico")
print(f"\nApós preencher, execute as Seções 4.3 (sugestão) e 5 (carregamento dos cortes).")


✓ Catálogo já existe e está atualizado  (426 features)

⚠  PRÓXIMA AÇÃO: abra features_diagnostico.xlsx e preencha as colunas:
   - label              : nome interpretável para o dashboard
   - tipo_feature       : contínua | binária | categórica | percentual | desvio_padrao
   - cortes_finais      : valores de corte separados por vírgula (ex: 0, 1000, 3000)
   - incluir_diagnostico: 0 para excluir da base diagnóstico

Após preencher, execute as Seções 4.3 (sugestão) e 5 (carregamento dos cortes).


## 4 · Funções de Faixamento

### 4.1 · Funções Auxiliares

In [33]:
# ─────────────────────────────────────────────────────────────────
# Arredondamento de cortes para labels mais limpos
# ─────────────────────────────────────────────────────────────────

def count_decimal_places(numero):
    numero_str = str(abs(numero))
    if int(numero) == numero:
        return 0
    return len(numero_str) - numero_str.index('.') - 1


def count_alg_integer(numero):
    return len(str(abs(int(numero))))


def round_2_alg(numero, round_algs=2):
    """Arredonda um número deixando apenas `round_algs` algarismos significativos.
    
    Exemplo: round_2_alg(1342.7, 2) → 1300.0
    """
    if numero == 0:
        return 0.0
    exp = count_decimal_places(numero)
    fator = 10 ** exp
    numero_int = numero * fator
    round_arg = count_alg_integer(numero_int) - round_algs
    return round(numero_int, -round_arg) / fator


# ─────────────────────────────────────────────────────────────────
# Formatação de labels para variáveis percentuais
# ─────────────────────────────────────────────────────────────────

def perc_faixas(label_faixa):
    """Converte cortes decimais no label para formato percentual.
    
    Exemplo: '2. De 0.1 até 0.3' → '2. De 10.0% até 30.0%'
    """
    if pd.isna(label_faixa):
        return None
    label_faixa = str(label_faixa)
    index = label_faixa.split('.', 1)[0]
    string = label_faixa.split('.', 1)[-1]
    numbers = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", string)
    for n in numbers:
        perc = f'{round(float(n) * 100, 1)}%'
        string = (' ' + string + ' ').replace(f' {n} ', f' {perc} ', 1).strip()
    return index + '.' + string


print("Funções auxiliares de faixamento definidas ✓")


Funções auxiliares de faixamento definidas ✓


### 4.2 · Faixador Principal (`faixar_diag`)

Versão unificada das funções de faixamento, com suporte a três modos de uso:

| Modo | Quando usar | Como configurar |
|---|---|---|
| **Zero separado** | Variáveis com zero estrutural (ausências, faltas, turnarounds) | Incluir `0` em `cortes` |
| **Zero incluído** | Variáveis contínuas onde zero é parte natural do range | Não incluir `0` em `cortes` |
| **Percentual** | Taxas e proporções (escala 0–1) | Passar `is_perc=True` |

**Labels gerados:**
- `"01. Zero"` / `"02. Acima de 0 até X"` / `"03. De X até Y"` / `"N. Y ou acima"` (zero separado)  
- `"1. Abaixo de X"` / `"2. De X até Y"` / `"N. Y ou acima"` (padrão)

Índices são zero-padded para 2 dígitos quando o total de faixas excede 9 (garante ordenação correta no Power BI/Tableau).


In [34]:
def faixar_diag(series: pd.Series, cortes: list, is_perc: bool = False) -> pd.Series:
    """
    Faixeia uma série numérica com labels textuais interpretáveis para dashboards.

    Dois modos de zero:
    - Zero em cortes  → zero recebe faixa própria ("N. Zero").
      Demais cortes definem intervalos para valores ≠ 0 (pode incluir negativos).
    - Zero fora de cortes → faixamento sequencial padrão, zero cai em alguma faixa.

    Indexação: zero-padded para 2 dígitos quando total de faixas > 9.
    NaN é sempre preservado como NaN.

    Args:
        series (pd.Series):  Série numérica a faixear.
        cortes (list):       Valores de corte. Incluir 0 para criar faixa própria de zero.
        is_perc (bool):      Se True, converte números nos labels para % (escala 0–1).

    Returns:
        pd.Series: Série com labels textuais. NaN mantido como NaN.

    Exemplos:
        faixar_diag(s, [0, 500, 1500])        → "1. Zero" | "2. Acima de 0 até 500" | ...
        faixar_diag(s, [500, 1500])            → "1. Abaixo de 500" | "2. De 500 até 1500" | ...
        faixar_diag(s, [0, 0.1, 0.3], True)   → "1. Zero" | "2. Acima de 0% até 10.0%" | ...
    """
    vals     = series.astype(float)
    result   = pd.Series(np.nan, index=series.index, dtype=object)
    not_null = vals.notna()

    cuts          = sorted({float(c) for c in cortes})
    has_zero_bin  = 0.0 in cuts
    non_zero_cuts = [c for c in cuts if c != 0.0]

    n_faixas = len(non_zero_cuts) + 1 + (1 if has_zero_bin else 0)
    fmt      = (lambda n: f"{n:02d}") if n_faixas > 9 else str

    def lbl(n, text):
        return f"{fmt(n)}. {text}"

    if has_zero_bin:
        result[not_null & (vals == 0)] = lbl(1, "Zero")
        idx       = 2
        neg_cuts  = sorted(c for c in non_zero_cuts if c < 0)
        pos_cuts  = sorted(c for c in non_zero_cuts if c > 0)

        # Lado negativo (ex: variações de headcount, deltas salariais)
        if neg_cuts:
            result[not_null & (vals < neg_cuts[0])] = lbl(idx, f"Abaixo de {neg_cuts[0]}")
            idx += 1
            for i in range(1, len(neg_cuts)):
                m = not_null & (vals >= neg_cuts[i - 1]) & (vals < neg_cuts[i])
                result[m] = lbl(idx, f"De {neg_cuts[i - 1]} até {neg_cuts[i]}")
                idx += 1
            result[not_null & (vals >= neg_cuts[-1]) & (vals < 0)] = lbl(
                idx, f"De {neg_cuts[-1]} até 0"
            )
            idx += 1

        # Lado positivo
        if pos_cuts:
            result[not_null & (vals > 0) & (vals < pos_cuts[0])] = lbl(
                idx, f"Acima de 0 até {pos_cuts[0]}"
            )
            idx += 1
            for i in range(1, len(pos_cuts)):
                m = not_null & (vals >= pos_cuts[i - 1]) & (vals < pos_cuts[i])
                result[m] = lbl(idx, f"De {pos_cuts[i - 1]} até {pos_cuts[i]}")
                idx += 1
            result[not_null & (vals >= pos_cuts[-1])] = lbl(idx, f"{pos_cuts[-1]} ou acima")
        else:
            # Apenas zero e cortes negativos — positivos vão para bin "Acima de 0"
            result[not_null & (vals > 0)] = lbl(idx, "Acima de 0")
    else:
        idx = 1
        result[not_null & (vals < cuts[0])] = lbl(idx, f"Abaixo de {cuts[0]}")
        idx += 1
        for i in range(1, len(cuts)):
            m = not_null & (vals >= cuts[i - 1]) & (vals < cuts[i])
            result[m] = lbl(idx, f"De {cuts[i - 1]} até {cuts[i]}")
            idx += 1
        result[not_null & (vals >= cuts[-1])] = lbl(idx, f"{cuts[-1]} ou acima")

    if is_perc:
        result = result.apply(perc_faixas)

    return result


print("faixar_diag definida ✓")
print()
print("Modos de uso:")
print("  Zero separado   : faixar_diag(serie, [0, 500, 1500])")
print("  Padrão          : faixar_diag(serie, [500, 1500])")
print("  Percentual      : faixar_diag(serie, [0, 0.1, 0.3], is_perc=True)")
print("  Com negativos   : faixar_diag(serie, [-1000, 0, 1000])  → zero separado + lados neg/pos")


faixar_diag definida ✓

Modos de uso:
  Zero separado   : faixar_diag(serie, [0, 500, 1500])
  Padrão          : faixar_diag(serie, [500, 1500])
  Percentual      : faixar_diag(serie, [0, 0.1, 0.3], is_perc=True)
  Com negativos   : faixar_diag(serie, [-1000, 0, 1000])  → zero separado + lados neg/pos


In [35]:
def dataframe_faixador(df: pd.DataFrame, dict_faixas: dict) -> pd.DataFrame:
    """
    Aplica faixar_diag a todas as colunas presentes em dict_faixas.

    A detecção de variáveis percentuais é feita automaticamente pelo padrão
    de nome (regex '_pc_' ou '^pc_'). Para forçar, chame faixar_diag diretamente.

    Args:
        df (pd.DataFrame):   DataFrame com as features originais.
        dict_faixas (dict):  {feature: [corte1, corte2, ...]}

    Returns:
        pd.DataFrame: Cópia do DataFrame com colunas em dict_faixas faixeadas.
    """
    df_out        = df.copy()
    pc_vars       = [c for c in df.columns if re.search(r'_pc_|^pc_', c)]
    aplicadas     = []
    ausentes      = []
    faixas_vazias = []

    for col, cortes in dict_faixas.items():
        if col not in df_out.columns:
            ausentes.append(col)
            continue
        if not isinstance(cortes, list) or len(cortes) == 0:
            faixas_vazias.append(col)
            continue
        is_perc     = col in pc_vars
        df_out[col] = faixar_diag(df_out[col], cortes, is_perc=is_perc)
        aplicadas.append(col)

    print(f"Faixamento aplicado : {len(aplicadas)} colunas")
    if faixas_vazias:
        print(f"Faixas vazias       : {faixas_vazias}")
    if ausentes:
        print(f"Ausentes no df      : {ausentes}")
    return df_out


print("dataframe_faixador definida ✓")


dataframe_faixador definida ✓


### 4.3 · Sugestão Automática de Faixas

In [36]:
def obter_faixas_ref(
    df: pd.DataFrame,
    colunas: list,
    zero_faixa: bool = True,
    round_algs: int = 2,
) -> dict:
    """
    Calcula faixas de referência por quartis (Q1, Mediana, Q3) dos valores não-zero.
    Útil para gerar o primeiro rascunho do dicionário antes do ajuste manual.

    Para variáveis com zero estrutural (zero_faixa=True), inclui 0 como primeiro corte.
    Cortes duplicados após arredondamento são removidos automaticamente.

    Args:
        df (pd.DataFrame):   DataFrame de referência (usar base TT).
        colunas (list):      Features numéricas a calcular.
        zero_faixa (bool):   Se True e a coluna possui zeros, inclui 0 nos cortes.
        round_algs (int):    Algarismos significativos no arredondamento dos cortes.

    Returns:
        dict: {feature: [corte1, corte2, ...]}
    """
    dist = {}
    for var in colunas:
        if var not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[var]):
            continue

        series  = df[var].dropna()
        nonzero = series[series != 0]

        if nonzero.empty:
            dist[var] = [0.0] if zero_faixa else []
            continue

        q1  = nonzero.quantile(0.25)
        med = nonzero.quantile(0.50)
        q3  = nonzero.quantile(0.75)

        cortes_raw = [q1, med, q3]
        cortes     = sorted({round_2_alg(c, round_algs) for c in cortes_raw if c != 0})

        if zero_faixa and (series == 0).any():
            cortes = [0.0] + cortes

        dist[var] = cortes

    return dist


print("obter_faixas_ref definida ✓")
print()
print("Uso:")
print("  dict_ref = obter_faixas_ref(df_base, feature_cols_numericas)")
print("  # → {feature: [0.0, Q1, mediana, Q3]}")


obter_faixas_ref definida ✓

Uso:
  dict_ref = obter_faixas_ref(df_base, feature_cols_numericas)
  # → {feature: [0.0, Q1, mediana, Q3]}


## 5 · Carregamento do Dicionário de Features (`features_diagnostico_classificado.xlsx`)

In [37]:

import ast

# ════════════════════════════════════════════════════════════════
# Lê features_diagnostico_classificado.xlsx (preenchido manualmente).
# Colunas usadas:
#   feature              → nome original da coluna na base
#   label                → nome interpretável para o dashboard
#   cortes_sugeridos     → lista de cortes no formato "[0, 10, 20, 50]"
#   incluir_diagnostico  → 1 = incluir | 0 = excluir
#
# Resultado:
#   features_incluir    → lista de features a incluir (incluir=1)
#   dict_cortes_final   → {feature: [cortes]}  (apenas as que têm corte)
#   features_sem_corte  → features incluídas mas sem corte → valor original mantido
#   dict_labels         → {feature: label}
# ════════════════════════════════════════════════════════════════
cat_path = REPORTS / "features_diagnostico_classificado.xlsx"

if not cat_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {cat_path}")

cat_df    = pd.read_excel(cat_path)
cat_ativos = cat_df[cat_df["incluir_diagnostico"] == 1].copy()


def parse_cortes(valor):
    """Converte string '[0, 10, 20, 50]' ou '0, 10, 20, 50' em lista de floats."""
    if pd.isna(valor) or str(valor).strip() in ("", "nan"):
        return None
    s = str(valor).strip()
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return [float(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass
    try:
        return [float(x.strip()) for x in s.split(",")]
    except ValueError:
        return None


dict_cortes_final  = {}
dict_labels        = {}
features_incluir   = []
features_sem_corte = []

for _, row in cat_ativos.iterrows():
    feat   = row["feature"]
    label  = str(row.get("label", "")).strip()
    cortes = parse_cortes(row.get("cortes_sugeridos"))

    features_incluir.append(feat)

    if label and label not in ("nan", ""):
        dict_labels[feat] = label

    if cortes is not None:
        dict_cortes_final[feat] = cortes
    else:
        features_sem_corte.append(feat)

print(f"Features ativas (incluir=1)  : {len(features_incluir)}")
print(f"Com cortes_sugeridos         : {len(dict_cortes_final)}")
print(f"Sem corte (valor original)   : {len(features_sem_corte)}")
print(f"Com label                    : {len(dict_labels)}")
print(f"\nFeatures sem corte (valor original mantido):")
for f in features_sem_corte:
    lbl = dict_labels.get(f, f)
    print(f"  {f:<35s}  →  {lbl}")


Features ativas (incluir=1)  : 345
Com cortes_sugeridos         : 329
Sem corte (valor original)   : 16
Com label                    : 344

Features sem corte (valor original mantido):
  ds_agrupamento                       →  Agrupamento
  ds_cargo                             →  Cargo
  ds_centro_custo                      →  Centro de Custo
  ds_escolaridade                      →  Escolaridade
  ds_estado                            →  Estado
  ds_etnia                             →  Etnia
  ds_genero                            →  Gênero
  ds_gestor                            →  Gestor
  ds_grade                             →  Grade
  ds_nivel_cargo                       →  Nível do cargo
  ds_pcd                               →  PCD
  ds_pcd_detalhe                       →  PCD - Detalhe
  ds_segmento                          →  Segmento
  ds_tipo_turno                        →  Turno - tipo
  ds_uorg                              →  UORG
  qt_companheiro                       →  Com

In [38]:

# Amostra dos cortes parseados para validação visual
print("Amostra de cortes_sugeridos parseados (primeiros 15):")
for feat, faixas in list(dict_cortes_final.items())[:15]:
    lbl = dict_labels.get(feat, feat)
    print(f"  {feat:<55s}: {faixas}")


Amostra de cortes_sugeridos parseados (primeiros 15):
  vl_abs_dias_sum_12m                                    : [0.0, 10.0, 20.0, 50.0]
  vl_abs_dias_sum_3m                                     : [0.0, 10.0, 20.0, 50.0]
  vl_abs_dias_sum_6m                                     : [0.0, 10.0, 20.0, 50.0]
  vl_adicional_noturno_dias                              : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_max_3m                       : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_max_6m                       : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_mean_3m                      : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_mean_6m                      : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_min_3m                       : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_min_6m                       : [0.0, 2.0, 5.0, 10.0, 20.0]
  vl_adicional_noturno_dias_std_3m                       : [0.0, 0.5, 1.0, 2.0]
  vl_adi

## 6 · Construção da Base Diagnóstico

In [ ]:

# ════════════════════════════════════════════════════════════════
# Carrega base_full_raw.parquet — universo completo (todas as linhas
# da mensalizada, com rolling features já calculadas em 01_ingest,
# antes dos filtros do pipeline do modelo).
#
# Mantém apenas COL_ID + COL_DATA + features_incluir.
# Features ausentes ficam como NaN (robustez).
# ════════════════════════════════════════════════════════════════
df_full = pd.read_parquet(DATA_GOLD / "base_full_raw.parquet")
df_full[COL_DATA] = pd.to_datetime(df_full[COL_DATA])
df_full[COL_ID]   = df_full[COL_ID].astype(str)

available = [f for f in features_incluir if f in df_full.columns]
missing   = [f for f in features_incluir if f not in df_full.columns]
if missing:
    print(f"⚠ {len(missing)} features ausentes em base_full_raw (serão NaN):")
    for f in missing[:10]:
        print(f"    {f}")
    if len(missing) > 10:
        print(f"    ... e mais {len(missing) - 10}")

keep = [COL_ID, COL_DATA] + available
df_diag = df_full[keep].copy()

# Garantir que features ausentes apareçam como NaN
for f in missing:
    df_diag[f] = np.nan

print(f"\n{'─' * 50}")
print(f"Origem    : base_full_raw.parquet  (universo completo, sem filtros)")
print(f"Linhas    : {len(df_diag):,}")
print(f"Colunas   : {df_diag.shape[1]}  ({len(available)} features + 2 meta)")
print(f"IDs únicos: {df_diag[COL_ID].nunique():,}")
print(f"Período   : {df_diag[COL_DATA].min():%Y-%m} → {df_diag[COL_DATA].max():%Y-%m}")



Partição    Linhas  Período
──────────────────────────────────────────────────
TT       106,902  2024-01 → 2025-05
Val       24,912  2025-06 → 2025-08
Apl       37,404  2025-09 → 2025-12
──────────────────────────────────────────────────
Total    169,218  348 colunas  |  13,532 IDs únicos


In [40]:

# ════════════════════════════════════════════════════════════════
# 1. Aplica faixamento nas features com cortes definidos
# 2. Mantém valor original nas features sem corte
# 3. Renomeia todas as colunas de features para os labels do dicionário
# 4. Salva apenas [COL_ID, COL_DATA] + features (sem _particao e sem meta)
# ════════════════════════════════════════════════════════════════

# ── Faixamento ────────────────────────────────────────────────
df_diagnostico = dataframe_faixador(df_diag, dict_cortes_final)

# ── Renomear features → labels ─────────────────────────────────
rename_dict = {
    feat: label
    for feat, label in dict_labels.items()
    if feat in df_diagnostico.columns
}
df_diagnostico = df_diagnostico.rename(columns=rename_dict)
print(f"✓ {len(rename_dict)} colunas renomeadas conforme labels do dicionário")

# ── Selecionar colunas finais: apenas ID + data + features ─────
# Converte features_incluir para os labels onde houver renomeação
cols_features_final = [
    dict_labels.get(f, f)
    for f in features_incluir
    if dict_labels.get(f, f) in df_diagnostico.columns or f in df_diagnostico.columns
]
# Garantir que a coluna realmente existe (após rename)
cols_features_final = [c for c in cols_features_final if c in df_diagnostico.columns]

cols_saida = [COL_ID, COL_DATA] + cols_features_final
df_diagnostico = df_diagnostico[cols_saida]

# ── Salvar ─────────────────────────────────────────────────────
out_path = DATA_GOLD / "base_diagnostico.parquet"
df_diagnostico.to_parquet(out_path, index=False)

mb = out_path.stat().st_size / 1e6
print(f"\n✓ base_diagnostico.parquet salva")
print(f"  Shape      : {df_diagnostico.shape[0]:,} linhas × {df_diagnostico.shape[1]} colunas")
print(f"  IDs únicos : {df_diagnostico[COL_ID].nunique():,}")
print(f"  Colunas    : {COL_ID} + {COL_DATA} + {len(cols_features_final)} features")
print(f"  Tamanho    : {mb:.1f} MB")
print(f"  Salvo em   : {out_path}")

# ── Amostra de distribuição ────────────────────────────────────
print(f"\n{'─' * 50}")
print("Amostra — distribuição de uma coluna faixeada:")
ex_feat  = list(dict_cortes_final.keys())[0]
ex_label = dict_labels.get(ex_feat, ex_feat)
if ex_label in df_diagnostico.columns:
    vc = df_diagnostico[ex_label].value_counts(dropna=False).sort_index()
    print(f"  {ex_label}:")
    print(vc.to_string())


Faixamento aplicado : 329 colunas
✓ 344 colunas renomeadas conforme labels do dicionário

✓ base_diagnostico.parquet salva
  Shape      : 169,218 linhas × 347 colunas
  IDs únicos : 13,532
  Colunas    : id_colaborador + dt_mes_ano + 345 features
  Tamanho    : 14.2 MB
  Salvo em   : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_diagnostico.parquet

──────────────────────────────────────────────────
Amostra — distribuição de uma coluna faixeada:
  Dias de absenteísmo (soma - 12 meses):
Dias de absenteísmo (soma - 12 meses)
1. Zero                   60860
2. Acima de 0 até 10.0    84979
3. De 10.0 até 20.0       16184
4. De 20.0 até 50.0        6337
5. 50.0 ou acima            289
NaN                         569
